In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Grafiklerin notebook içinde görünmesi için
%matplotlib inline

In [ ]:
##dataframe
df = pd.read_csv("data/europe_air_routes.csv")

In [ ]:
#EDA(Exploratory Data Analysis)- Veri Keşfi
df.head()
df.info()
df.columns

In [ ]:
#EDA - Veri Keşfi
print(f"Satır Sayısı: {df.shape[0]}")
print(f"Sütun Sayısı: {df.shape[1]}")


In [ ]:
#İlk 5 satır
df.head()

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing.sort_values(ascending=False)

Veri setindeki eksik değer oranı %1'in altındadır ve sosyal ağın oluşturulmasında kullanılan IATA kodları (iata_from, iata_to) eksiksiz olduğundan analiz sonuçlarını etkilememektedir.

In [ ]:
df.describe()

In [ ]:
airports = set(df["iata_from"]).union(set(df["iata_to"]))

print(f"Toplam Havalimanı Sayısı: {len(airports)}")

In [ ]:
##Toplam rota sayısı
print("Toplam rota :" , len(df))

In [ ]:
##En çok kalkış yapılan havalimanları
top_departure_airports = df["iata_from"].value_counts().head(10)
print("En çok kalkış yapılan havalimanları:")
print(top_departure_airports)

In [ ]:
##En çok varış yapılan havalimanları
top_arrival_airports = df["iata_to"].value_counts().head(10)

In [ ]:
##En yoğun şehirler
top_cities_arv = df["arrival_airport_city_name"].value_counts().head(10)
top_cities_dep = df["departure_city"].value_counts().head(10)



In [ ]:
##Fşyat dağılımı
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["price"], bins=30)
plt.title("Ticket Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
##Haftalık uçuş dağılımı
plt.figure(figsize=(8,5))
plt.hist(df["flights_per_week"], bins=20)
plt.title("Flights Per Week")
plt.xlabel("Flights")
plt.ylabel("Frequency")
plt.show()

In [ ]:
top10 = df["iata_from"].value_counts().head(10)

plt.figure(figsize=(10,5))

top10.plot(kind="bar")

plt.title("Top 10 Departure Airports")

plt.ylabel("Number of Routes")

plt.savefig("top10_departure_airports.png", dpi=300, bbox_inches="tight")

plt.show()



In [ ]:
top10 = df["iata_to"].value_counts().head(10)

plt.figure(figsize=(10,5))

top10.plot(kind="bar")

plt.title("Top 10 Arrival Airports")

plt.ylabel("Number of Routes")

plt.show()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(df["iata_from"].isnull().sum())
print(df["iata_to"].isnull().sum())

In [ ]:
df["flights_per_week"].dtype

In [ ]:
print((df["price"] < 0).sum())
print((df["common_duration"] <= 0).sum())

The dataset was checked for duplicate records, missing values, incorrect data types, and invalid numerical values. No duplicate records or missing IATA airport codes were found. Missing values existed only in a few geographic attributes, which were not required for network construction. Therefore, no records were removed during the cleaning process.

Explotary Data Analysis part (EDA) is over.

--------------------------------------------------------

NETWORK CONSTRUCTION


In [ ]:
##NETWORK CONSTRUCTION

G = nx.from_pandas_edgelist(
    df,
    source="iata_from",
    target="iata_to",
    edge_attr=[
        "price",
        "flights_per_week",
        "common_duration"
    ],
    create_using=nx.DiGraph()
)

In [ ]:
##Undirected version - just in case we want to analyze the network as undirected
G_undirected = G.to_undirected()

In [ ]:
##Attributes of the graph
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Network density:", nx.density(G))
print("Is strongly connected:", nx.is_strongly_connected(G))
print("Is weakly connected:", nx.is_weakly_connected(G))



In [ ]:
##Ortalama Degree
avg_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()

print(avg_degree)

Graph construction complete. Vertex and edge numbers calculated. Network density calculated. Strong and weak connectivity checked.


-------------------------------

SOCIAL NETWORK ANALYSIS


In [ ]:
##Use undirected graph for degree centrality
degree_centrality = nx.degree_centrality(G_undirected)

##Convert to DataFrame for easier analysis
degree_df = pd.DataFrame(
    degree_centrality.items(),
    columns=["Airport", "Degree_Centrality"]
)

##Sort
degree_df = degree_df.sort_values(
    by="Degree_Centrality", ascending=False
)





In [ ]:
###Top 10 airports by degree centrality
top10_degree = degree_df.head(10)
print(top10_degree)

In [ ]:
##Visualize
top10 = degree_df.head(10)

plt.figure(figsize=(10,5))

plt.bar(top10["Airport"], top10["Degree_Centrality"])

plt.title("Top 10 Airports by Degree Centrality")

plt.xlabel("Airport")

plt.ylabel("Degree Centrality")

plt.xticks(rotation=45)

plt.show()

Airports with the highest degree centrality have the largest number of direct connections within the European airport transportation network.

In [ ]:
airport_info = (
    df[["iata_to", "arrival_airport_name", "arrival_airport_city_name"]]
    .drop_duplicates(subset="iata_to")
)

degree_df = degree_df.merge(
    airport_info,
    left_on="Airport",
    right_on="iata_to",
    how="left"
)

degree_df = degree_df[[
    "Airport",
    "arrival_airport_name",
    "arrival_airport_city_name",
    "Degree_Centrality"
]]

degree_df.head(10)

In [ ]:
##IN-DEGREE OUT-DEGREE
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

In [ ]:
##Create df
in_degree_df = pd.DataFrame(
    in_degree.items(),
    columns=["Airport", "In_Degree"]
).sort_values(
    by="In_Degree", ascending=False
)

out_degree_df = pd.DataFrame(
    out_degree.items(),
    columns=["Airport", "Out_Degree"]
).sort_values(
    by="Out_Degree", ascending=False
)

In [ ]:
##top10
print("Top 10 In-Degree Airports")
display(in_degree_df.head(10))

print("Top 10 Out-Degree Airports")
display(out_degree_df.head(10))

In [ ]:
##In-Degree Visualization
top10 = in_degree_df.head(10)

plt.figure(figsize=(10,5))

plt.bar(top10["Airport"], top10["In_Degree"])

plt.title("Top 10 Airports by In-Degree")

plt.xlabel("Airport")

plt.ylabel("In-Degree")

plt.xticks(rotation=45)

plt.show()

In [ ]:
#Out-Degree Visualization
top10 = out_degree_df.head(10)

plt.figure(figsize=(10,5))

plt.bar(top10["Airport"], top10["Out_Degree"])

plt.title("Top 10 Airports by Out-Degree")

plt.xlabel("Airport")

plt.ylabel("Out-Degree")

plt.xticks(rotation=45)

plt.show()

In [ ]:
airport_info = (
    df[["iata_to",
        "arrival_airport_name",
        "arrival_airport_city_name"]]
    .drop_duplicates("iata_to")
)